In [90]:
import numpy as np
import pandas as pd
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)

import warnings
warnings.filterwarnings('ignore')
RANDOM_STATE = 100

## Data splitting

In [91]:
def split_data(path, delta=True):
    #Load file
    data = pd.read_csv(path)
    
    #Get features
    feature_names = [col for col in data.columns if col not in ['label', 'type', 'person_id', 'recording']]
    if not delta:
        feature_names = [col for col in feature_names if not col.startswith('d')]


    X = data[feature_names].values
    y = data['type'].values

    #Train/test split (80/20)
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y
    )

    print(f"Training: {X_train.shape[0]} samples")
    print(f"Test: {X_test.shape[0]} samples")
    print(f"""Train distribution:
        - Hyperfunktionelle Dysphonie: {sum(y_train=='Hyperfunktionelle Dysphonie')}
        - Rekurrensparese: {sum(y_train=='Rekurrensparese')}
        - Laryngitis: {sum(y_train=='Laryngitis')}
        """)

    #Feature scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return {
        "X_train": X_train,
        "X_test": X_test,
        "X_train_scaled": X_train_scaled,
        "X_test_scaled": X_test_scaled,
        "y_train": y_train,
        "y_test": y_test,
        "scaler": scaler
    }

In [92]:
rec_type = 'phrase'
# rec_type = 'a_n'
# rec_type = 'i_n'
# rec_type = 'u_n'

In [93]:
data = split_data(f'data/preped/multi-{rec_type}.csv', delta=True)

Training: 444 samples
Test: 112 samples
Train distribution:
        - Hyperfunktionelle Dysphonie: 169
        - Rekurrensparese: 169
        - Laryngitis: 106
        


## SVM

In [74]:
def train_multiclass_svm(data, strategy, kernel='rbf', C=10, sigma=1):
    
    base_svm = SVC(
        C=C,
        kernel=kernel,
        gamma=1 / (2 * (sigma ** 2)),
        # gamma='scale',
        random_state=RANDOM_STATE,
        probability=True,
        class_weight='balanced'
    )
    
    if strategy == 'ovr':
        model = OneVsRestClassifier(base_svm)
    elif strategy == 'ovo':
        model = OneVsOneClassifier(base_svm)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")
    
    # Full training
    model.fit(data['X_train_scaled'], data['y_train'])
    training_accuracy = model.score(data['X_train_scaled'], data['y_train'])
    
    # Metrics on test set
    y_pred = model.predict(data['X_test_scaled'])
    accuracy = accuracy_score(data['y_test'], y_pred)
    precision = precision_score(data['y_test'], y_pred, average='weighted')
    recall = recall_score(data['y_test'], y_pred, average='weighted')
    f1 = f1_score(data['y_test'], y_pred, average='weighted')
    
    return {
        'model': model,
        'y_pred': y_pred,
        'training_accuracy': training_accuracy,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

In [75]:
strategies = ['ovr', 'ovo']
kernel_types = ['rbf']
# c_values = [0.1, 1]
c_values = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]

# sigma_values = [1, 4, 10, 50, 100]
sigma_values = [5, 6, 7, 8, 10, 12, 15, 20, 25, 30]

results = []
trained_models = {}

for strategy in strategies:
    for kernel in kernel_types:
        for C in c_values:

            sigmas = sigma_values if kernel == 'rbf' else [None]

            for sigma in sigmas:

                name = f"{rec_type}-{strategy}-{C}-{sigma}"
    
                """
                print(
                    f"kernel={kernel}, C={C}" +
                    (f", sigma={sigma}" if sigma is not None else ""),
                    end=' '
                )
                """

                kwargs = {
                    'strategy': strategy,
                    'kernel': kernel,
                    'C': C
                }

                if sigma is not None:
                    kwargs['sigma'] = sigma

                result = train_multiclass_svm(
                    data,
                    **kwargs
                )

                trained_models[name] = result

                results.append({
                    'model': name,
                    'training_accuracy': result['training_accuracy'],
                    'accuracy': result['accuracy'],
                    'precision': result['precision'],
                    'recall': result['recall'],
                    'f1_score': result['f1_score']
                })

print("\nTraining complete.")


Training complete.


In [76]:
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

             model  training_accuracy  accuracy  precision   recall  f1_score
 phrase-ovr-0.01-5           0.725225  0.482143   0.573732 0.482143  0.488946
 phrase-ovr-0.01-6           0.635135  0.491071   0.592894 0.491071  0.500865
 phrase-ovr-0.01-7           0.601351  0.500000   0.599524 0.500000  0.510165
 phrase-ovr-0.01-8           0.581081  0.526786   0.630642 0.526786  0.533043
phrase-ovr-0.01-10           0.542793  0.508929   0.641568 0.508929  0.516421
phrase-ovr-0.01-12           0.515766  0.446429   0.694792 0.446429  0.444222
phrase-ovr-0.01-15           0.461712  0.357143   0.672880 0.357143  0.314265
phrase-ovr-0.01-20           0.403153  0.330357   0.673611 0.330357  0.272442
phrase-ovr-0.01-25           0.394144  0.321429   0.664876 0.321429  0.261630
phrase-ovr-0.01-30           0.389640  0.312500   0.664239 0.312500  0.244180
 phrase-ovr-0.03-5           0.632883  0.517857   0.399926 0.517857  0.449027
 phrase-ovr-0.03-6           0.574324  0.508929   0.393927 0.508

## Random Forest

In [81]:
def train_multiclass_rf(data, n_estimators=100, max_depth=None):

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    # Train on full training set
    model.fit(data['X_train'], data['y_train'])

    training_accuracy = model.score(
        data['X_train'],
        data['y_train']
    )

    # Test metrics
    y_pred = model.predict(data['X_test'])

    accuracy = accuracy_score(data['y_test'], y_pred)
    precision = precision_score(data['y_test'], y_pred, average='weighted')
    recall = recall_score(data['y_test'], y_pred, average='weighted')
    f1 = f1_score(data['y_test'], y_pred, average='weighted')

    return {
        'model': model,
        'y_pred': y_pred,
        'training_accuracy': training_accuracy,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

In [88]:
n_estimators_values = [5, 10, 50, 100, 150]
max_depth_values = [10, 20]

results = []
trained_models = {}

for n_estimators in n_estimators_values:
    for max_depth in max_depth_values:

        name = f"RFM-{n_estimators}-{max_depth}"
        result = train_multiclass_rf(
            data,
            n_estimators=n_estimators,
            max_depth=max_depth
        )

        trained_models[name] = result

        results.append({
            'model': name,
            'training_accuracy': result['training_accuracy'],
            'accuracy': result['accuracy'],
            'precision': result['precision'],
            'recall': result['recall'],
            'f1_score': result['f1_score']
        })

print("\nTraining complete.")


Training complete.


In [89]:
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

     model  training_accuracy  accuracy  precision   recall  f1_score
  RFM-5-10           0.934685  0.455357   0.457389 0.455357  0.456320
  RFM-5-20           0.957207  0.419643   0.430166 0.419643  0.415194
 RFM-10-10           0.986486  0.553571   0.550460 0.553571  0.551806
 RFM-10-20           0.995495  0.455357   0.471015 0.455357  0.461577
 RFM-50-10           1.000000  0.535714   0.528942 0.535714  0.527137
 RFM-50-20           1.000000  0.482143   0.470938 0.482143  0.472886
RFM-100-10           1.000000  0.526786   0.514330 0.526786  0.517741
RFM-100-20           1.000000  0.508929   0.496700 0.508929  0.495989
RFM-150-10           1.000000  0.517857   0.506494 0.517857  0.509394
RFM-150-20           1.000000  0.517857   0.512177 0.517857  0.507313


## Logistic regression

In [98]:
def train_multiclass_lr(data, max_iter=1000):
    """Train multinomial logistic regression (softmax regression)."""

    model = LogisticRegression(
        multi_class='multinomial',  # softmax regression
        solver='lbfgs',
        random_state=RANDOM_STATE,
        class_weight='balanced',
        max_iter=max_iter,
        n_jobs=-1
    )

    # Train model
    model.fit(data['X_train_scaled'], data['y_train'])

    training_accuracy = model.score(
        data['X_train_scaled'], data['y_train']
    )

    # Predictions
    y_pred = model.predict(data['X_test_scaled'])

    # Metrics
    accuracy = accuracy_score(data['y_test'], y_pred)
    precision = precision_score(data['y_test'], y_pred, average='weighted')
    recall = recall_score(data['y_test'], y_pred, average='weighted')
    f1 = f1_score(data['y_test'], y_pred, average='weighted')

    print('training_accuracy=', training_accuracy)
    print('accuracy=', accuracy)
    print('precision=', precision)
    print('recall=', recall)
    print('f1_score=', f1)
    
    return {
        'model': model,
        'y_pred': y_pred,
        'training_accuracy': training_accuracy,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

In [99]:
print(f'{rec_type}:')
x = train_multiclass_lr(data)

phrase:
training_accuracy= 0.713963963963964
accuracy= 0.5178571428571429
precision= 0.5508583838940982
recall= 0.5178571428571429
f1_score= 0.5262206543456543
